In [3]:
# imports
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import pandas as pd
from backtesting import Backtest
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator

from datetime import timedelta

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [4]:
# swing structure
def implement_market_structure_states_strict(df, n=5):
    """Calcule les états du marché avec transition obligatoire par l'état NEUTRAL.

    Aucun passage direct entre BULLISH et BEARISH n'est autorisé.
    """
    df = df.copy()
    length = len(df)

    df['is_swing_high'] = False
    df['is_swing_low'] = False
    for k in range(n, len(df) - n):
        # Le plus haut des n bougies précédentes et suivantes
        high_range = df['High'].iloc[k - n:k + n + 1]
        if df['High'].iloc[k] == high_range.max():
            df.loc[df.index[k], 'is_swing_high'] = True
            
        # Le plus bas des n bougies précédentes et suivantes
        low_range = df['Low'].iloc[k - n:k + n + 1]
        if df['Low'].iloc[k] == low_range.min():
            df.loc[df.index[k], 'is_swing_low'] = True

    states = []
    current_state = 'NEUTRAL'

    # Variables de structure de marché
    last_confirmed_high = None
    last_confirmed_low = None
    higher_low = None  # Le plancher (uniquement actif en BULLISH)
    lower_high = None  # Le plafond (uniquement actif en BEARISH)

    for j in range(length):
        close_p = df["Close"].iloc[j]

        # --- ÉTAPE B : Machine à états avec transit strict par NEUTRAL ---
        if current_state == "BULLISH":
            # Sortie de Bullish : Clôture sous le Higher Low (HL) -> NEUTRAL uniquement
            if higher_low is not None and close_p < higher_low:
                current_state = "NEUTRAL"
                higher_low = None  # Reset des repères obsolètes
                lower_high = None

        elif current_state == "BEARISH":
            # Sortie de Bearish : Clôture au-dessus du Lower High (LH) -> NEUTRAL uniquement
            if lower_high is not None and close_p > lower_high:
                current_state = "NEUTRAL"
                higher_low = None
                lower_high = None  # Reset des repères obsolètes

        elif current_state == "NEUTRAL":
            # En zone neutre, on attend un NOUVEAU BOS pour définir la tendance
            # Entrée en Bullish : Clôture au-dessus du dernier sommet confirmé
            # il faut tjs un high and un low avant de former une structure bearish ou bullish
            if last_confirmed_high is not None and last_confirmed_low is not None and close_p > last_confirmed_high:
                current_state = "BULLISH"
                higher_low = (
                    last_confirmed_low  # On fige le dernier creux comme HL
                )

            # Entrée en Bearish : Clôture sous le dernier creux confirmé
            elif last_confirmed_high is not None and last_confirmed_low is not None and close_p < last_confirmed_low:
                current_state = "BEARISH"
                lower_high = (
                    last_confirmed_high  # On fige le dernier sommet comme LH
                )

        states.append(current_state)

         # --- ÉTAPE A : Enregistrement des Swings dès qu'ils se confirment ---
        if df["is_swing_high"].iloc[j] == True:
            last_confirmed_high = df["High"].iloc[j]
            if current_state == "BEARISH" and lower_high > last_confirmed_high:
                lower_high = last_confirmed_high

        if df["is_swing_low"].iloc[j] == True:
            last_confirmed_low = df["Low"].iloc[j]
            if current_state == "BULLISH" and higher_low < last_confirmed_low:
                higher_low = last_confirmed_low

    df["market_state"] = states
    return df


In [5]:
# rsi trending stratey + m15 structure
from backtesting import Strategy

global_tp_multiple_rsi = 0
global_stop_trade_rsi = 0
global_start_trade_hour = 0
global_er15_threshold = 0

class OpeningRangeBreakout(Strategy):
    # variables
    tp_multiple_rsi = 5
    start_trade_hour = 5
    stop_trade_rsi = 16
    er15_threshold = 0.3

    # fixed variables
    risk_percent = 0.01
    max_leverage = 4
    my_equity = 10000

    # def init(self, range_to_consider, tp_multiple, stop_trade_hour, close_range_hour):
    def init(self):
        self.current_day = None
        self.current_day_order_placed = False
        self.last_low = None
        self.last_high = None
        self.order_placed_hour = None
        self.order_executed = False
        self.is_rsi_touched = False

        # when rsi is touched then we want trend to switch to bullish/bearish
        self.transition_state = None

        if global_tp_multiple_rsi != 0:
            self.tp_multiple_rsi = global_tp_multiple_rsi

        if global_stop_trade_rsi != 0:
            self.stop_trade_rsi = global_stop_trade_rsi

        if global_start_trade_hour != 0:
            self.start_trade_hour = global_start_trade_hour

        if global_er15_threshold != 0:
            self.er15_threshold = global_er15_threshold

    def reset_range(self, day):
        # delete placed order at the end of the day if not executed
        if self.current_day_order_placed is True and self.order_placed_hour:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            if self.order_executed == False:
                for order in self.orders:
                    if order.tag == self.get_tag():
                        order.cancel()

        self.current_day = day
        self.current_day_order_placed = False
        self.order_placed_hour = None
        self.last_low = None
        self.last_high = None
        self.order_executed = False
        self.is_rsi_touched = False
        self.transition_state = None

    def get_tag(self):
        return f"{self.current_day} {self.order_placed_hour}"

    def place_long_order(self, candle_date):
        if self.last_high is not None:
            planned_entry_price = self.last_high
            sl_price = planned_entry_price - self.data['ATR'][-1]
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price + per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            # print(candle_date)
            # print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"long per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price > sl_price:
                # print(f'Go long limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.buy(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
        
    def place_short_order(self, candle_date):
        if self.last_low is not None:
            planned_entry_price = self.last_low
            sl_price = planned_entry_price + self.data['ATR'][-1]
            per_share_risk = abs(planned_entry_price - sl_price)
            tp_multiple_rsi = self.tp_multiple_rsi 
            tp_price = planned_entry_price - per_share_risk * tp_multiple_rsi
            position_size = int((self.risk_percent * self.equity) / per_share_risk)

            # print(candle_date)
            # print(f'limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')

            # print(f"short per_share_risk = {per_share_risk} & atr = {self.data['ATR'][-1]}")
            # if per_share_risk < self.data['ATR'][-1] * 5 and per_share_risk > self.data['ATR'][-1] * 1:
            if planned_entry_price < sl_price:
                # print(f'Go short limit = {planned_entry_price} tp = {tp_price} sl = {sl_price} position size = {position_size}')
                self.order_placed_hour = candle_date.time().hour
                self.sell(size = position_size, limit=planned_entry_price, tp=tp_price, sl=sl_price, tag=self.get_tag())
                self.current_day_order_placed = True
            
    def use_rsi_entry_strategy_market_ranging(self, candle_datetime):
        if self.is_rsi_touched == False:
            # print(f"EMA50 = {self.data['EMA_50'][-1]} & EMA200 = {self.data['EMA_200'][-1]} & RSI14 = {self.data['RSI14'][-1]}")
            self.is_rsi_touched = self.data['RSI14'][-1] < 30 and self.data['EMA_50'][-1] > self.data['EMA_200'][-1] or self.data['RSI14'][-1] > 70 and self.data['EMA_50'][-1] < self.data['EMA_200'][-1]

        if self.is_rsi_touched is not None:
            if self.data['RSI14'][-1] > 70 and self.data['EMA_50'][-1] > self.data['EMA_200'][-1] or self.data['RSI14'][-1] < 30 and self.data['EMA_50'][-1] < self.data['EMA_200'][-1]:
                self.is_rsi_touched = False
            
        c_long = self.data['EMA_50'][-1] > self.data['EMA_200'][-1] and self.data['market_state'][-1] == 'BULLISH'
        c_short = self.data['EMA_50'][-1] < self.data['EMA_200'][-1] and self.data['market_state'][-1] == 'BEARISH'

        # when rsi is touched then we want to see market state change from NEUTRAL to BULLISH/BEARISH
        c_transition = self.transition_state == 'NEUTRAL'

        if not self.current_day_order_placed and self.is_rsi_touched:
            # print(f"not placed RSI14 = {self.data['RSI14'][-1]}")
            if self.data['market_state'][-1] == 'NEUTRAL':
                self.transition_state = 'NEUTRAL'

            if c_long and c_transition:
                self.place_long_order(candle_datetime)

            if c_short and c_transition:
                self.place_short_order(candle_datetime)
        else:
            for trade in self.trades:
                if trade.tag == self.get_tag():
                    self.order_executed = True

            # print(f"placed RSI14 = {self.data['RSI14'][-1]}")
            # if self.order_executed == False:
            #     for order in self.orders:
            #         if order.tag == self.get_tag():
            #             order.cancel()
            #             self.current_day_order_placed = False
            #             if order.is_long:
            #                 self.place_long_order(candle_datetime)
            #             else:
            #                 self.place_short_order(candle_datetime)

    def next(self):
        candle_datetime = self.data.index[-1]
        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date or candle_datetime.time().hour >= self.stop_trade_rsi:
            self.reset_range(current_bar_date)

        # print(self.data['is_swing_low'].shift(2))
        if self.data['is_swing_low'][len(self.data['is_swing_low'])-3] == True:
            self.last_low = self.data.Low[len(self.data['is_swing_low'])-3]

        if self.data['is_swing_high'][len(self.data['is_swing_low'])-3] == True:
            self.last_high = self.data.High[len(self.data['is_swing_low'])-3]

        c_time_limit = candle_datetime.time().hour < self.stop_trade_rsi and candle_datetime.time().hour > self.start_trade_hour
        
        if c_time_limit:
            # print(candle_datetime)
            # print(f"c_market_structure = {self.data['market_state'][-1]} & c_er = {self.data['ER'][-1]}")
            self.use_rsi_entry_strategy_market_ranging(candle_datetime)

        # if self.current_day_order_placed == True:
        #     print(candle_datetime)
        #     print(self.orders)
        #     print(self.trades)
        # print("--------------------------------")
            



In [6]:
# variables
# all strategy
window = 2
global_er15_threshold = 0.3

# rsi
global_start_trade_hour = 7
global_tp_multiple_rsi = 2
global_stop_trade_rsi = 19

# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = '2026-1-1 00:00'
end_date = '2026-9-16 00:00'
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'
pair = 'GBPJPY'
timezone = "Europe/London"

In [ ]:
# load data

df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_h4_raw)
print(df_m15_raw)

                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.785    44503
2010-09-10 12:00:00  129.785  130.065  129.425  129.570    54127
2010-09-10 16:00:00  129.580  129.770  129.120  129.135    39740
2010-09-10 20:00:00  129.130  129.335  129.120  129.255     8320
...                      ...      ...      ...      ...      ...
2026-09-23 20:00:00  209.580  209.701  209.431  209.564    16661
2026-09-24 00:00:00  209.566  209.638  208.768  209.072    51880
2026-09-24 04:00:00  209.072  209.859  209.019  209.838    38121
2026-09-24 08:00:00  209.840  210.067  209.417  209.996    62442
2026-09-24 12:00:00  209.997  210.105  209.846  209.848    47545

[25029 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            

In [11]:
# process h4 data
df_h4 = df_h4_raw.loc['2021-10-1 00:00' : end_date]

indicator_ema_50 = EMAIndicator(df_h4['Close'], window=50)
df_h4['EMA_50'] = indicator_ema_50.ema_indicator()

indicator_ema_200 = EMAIndicator(df_h4['Close'], window=200)
df_h4['EMA_200'] = indicator_ema_200.ema_indicator()
print(df_h4)


                        Open     High      Low    Close  Tickvol      EMA_50  \
Date                                                                           
2021-10-01 00:00:00  150.099  150.168  149.594  149.683    20985         NaN   
2021-10-01 04:00:00  149.684  149.767  149.381  149.504    23262         NaN   
2021-10-01 08:00:00  149.506  150.328  149.226  150.248    43561         NaN   
2021-10-01 12:00:00  150.248  150.740  150.152  150.236    54285         NaN   
2021-10-01 16:00:00  150.236  150.687  150.216  150.526    26078         NaN   
...                      ...      ...      ...      ...      ...         ...   
2026-09-15 08:00:00  208.652  209.090  208.514  208.655    56407  209.590110   
2026-09-15 12:00:00  208.655  209.211  208.607  209.168    74480  209.573556   
2026-09-15 16:00:00  209.167  209.224  208.885  209.066    38410  209.553652   
2026-09-15 20:00:00  209.064  209.082  208.872  209.019    18776  209.532685   
2026-09-16 00:00:00  209.018  209.394  2

In [ ]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date : end_date]
df_m15 = implement_market_structure_states_strict(df_m15, n=window)


df_m15_periode = df_m15.loc['2026-06-5 10:00' : '2026-06-5 19:00']
# Affichage des transitions d'états
df_m15_periode["state_changed"] = df_m15_periode["market_state"] != df_m15_periode[
    "market_state"
].shift(1)
print(df_m15_periode[df_m15_periode["state_changed"]][["Close", "market_state"]])
# df_m15_periode2 = df_m15.loc['2026-06-5 13:00' : '2026-06-5 14:00']
# print(df_m15_periode2)
# for row in df_m15_periode[df_m15_periode["state_changed"]][["Close", "market_state"]].iterrows():
#     print(row)

m15_tf_ema_50_list = []
m15_tf_ema_200_list = []

for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        m15_tf_ema_50_list.append(df_h4.iloc[i-1]['EMA_50'])
        m15_tf_ema_200_list.append(df_h4.iloc[i-1]['EMA_200'])

df_m15['EMA_50'] = m15_tf_ema_50_list
df_m15['EMA_200'] = m15_tf_ema_200_list

indicator_atr_14 = AverageTrueRange(df_m15['High'], df_m15['Low'], df_m15['Close'], window=14)
df_m15['ATR'] = indicator_atr_14._atr

indicator_rsi_14 = RSIIndicator(df_m15['Close'], window=14)
df_m15['RSI14'] = indicator_rsi_14._rsi
df_m15.index = pd.to_datetime(df_m15.index, utc=True).tz_convert(timezone)

print(df_m15)

                       Close market_state
Date                                     
2026-06-05 10:00:00  215.330      BULLISH
2026-06-05 14:00:00  214.649      NEUTRAL
2026-06-05 16:15:00  214.351      BEARISH
                              Open     High      Low    Close  Tickvol  \
Date                                                                     
2026-01-01 23:00:00+00:00  210.689  210.871  210.574  210.850      439   
2026-01-01 23:15:00+00:00  210.850  210.984  210.574  210.982      133   
2026-01-01 23:30:00+00:00  210.982  210.982  210.720  210.909      340   
2026-01-01 23:45:00+00:00  210.909  211.068  210.909  211.068      355   
2026-01-02 00:00:00+00:00  211.067  211.347  211.018  211.270      715   
...                            ...      ...      ...      ...      ...   
2026-09-16 00:00:00+01:00  208.895  209.029  208.895  209.018     1853   
2026-09-16 00:15:00+01:00  209.020  209.027  208.982  208.996     1268   
2026-09-16 00:30:00+01:00  208.997  209.013  208.9

In [13]:
# run backtest
bt = Backtest(df_m15, OpeningRangeBreakout, margin=0.02, finalize_trades=True)

stats = bt.run()

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2026-04-15 18:15:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See https://github.com/kernc/backtesting.py/issues/119
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1039: UserWarning: (2026-07-02 17:00:00+01:00) A contingent SL/TP order would execute in the same bar its parent stop/limit order was turned into a trade. Since we can't assert the precise intra-candle price movement, the affected SL/TP order will instead be executed on the next (matching) price/bar, making the result (of this trade) somewhat dubious. See http

Start                     2026-01-01 23:00...
End                       2026-09-16 01:00...
Duration                    257 days 01:00:00
Exposure Time [%]                     1.99261
Equity Final [$]                  10469.43901
Equity Peak [$]                   11064.93898
Return [%]                            4.69439
Buy & Hold Return [%]                -0.84183
Return (Ann.) [%]                      6.4845
Volatility (Ann.) [%]                11.02992
CAGR [%]                              6.73594
Sharpe Ratio                           0.5879
Sortino Ratio                         1.15507
Calmar Ratio                          1.07164
Alpha [%]                             4.77954
Beta                                  0.10115
Max. Drawdown [%]                    -6.05099
Avg. Drawdown [%]                    -1.96849
Max. Drawdown Duration       93 days 19:00:00
Avg. Drawdown Duration       19 days 02:04:00
# Trades                                   37
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\bokeh\util\serialization.py:256: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1261', ...)